In [1]:
!stretch_free_robot_process.py
!stretch_robot_home.py

Done!
For use with S T R E T C H (R) from Hello Robot Inc.
---------------------------------------------------------------------

--------- Homing Head ----
--------- Homing Lift ----
Homing Lift...
Hardstop detected at motor position (rad) 115.88426971435547
Marking Lift position to 1.101578 (m)
Marking Lift position to 0.000000 (m)
[INFO] [robot_monitor]: Guarded contact lift
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
Lift homing successful
--------- Homing Arm ----
Homing Arm...
[INFO] [robot_monitor]: Wrist single tap: 1777
[INFO] [robot_monitor]: Wrist single tap: 1778
Hardstop detected at motor position (rad) 7.1129150390625
Marking Arm position to 0.000000 (m)
[INFO] [robot_monitor]: Guarded contact arm
[INFO] [robot_monitor]: Wrist single tap: 1779
[INFO] [robot_monitor]: Wrist single tap: 1780
[INFO] [robot_monitor]: Wrist

In [2]:
%pip install h5py

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 32.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import itertools
import time

import h5py
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

In [3]:
import stretch_body.robot

# ref: https://docs.hello-robot.com/0.3/python/moving/

robot = stretch_body.robot.Robot()

did_startup = robot.startup()
print(f"Robot connected to hardware: {did_startup}")

is_homed = robot.is_homed()
print(f"Robot is homed: {is_homed}")

Robot connected to hardware: True
Robot is homed: True
New collision pair event: link_gripper_s3_body_TO_link_arm_l1 [1772816921.5172498]
New collision pair event: link_gripper_s3_body_TO_link_arm_l1 [1772817187.56827]
New collision pair event: link_gripper_s3_body_TO_link_arm_l1 [1772817456.1061919]
New collision pair event: link_gripper_s3_body_TO_link_arm_l1 [1772817724.6425226]
New collision pair event: link_gripper_s3_body_TO_link_arm_l1 [1772817994.0547516]


In [ ]:
%pip install pykalman

# Presets

- stepper: slow / default / fast / max / trajectory_max
- dynamixel: default / max / trajectory_max

```python
r.arm.move_to(
    0.3,
    v_m=r.arm.params['motion']['max']['vel_m'],
    a_m=r.arm.params['motion']['max']['accel_m']
)

r.lift.move_to(
    0.6,
    v_m=r.lift.params['motion']['slow']['vel_m'],
    a_m=r.lift.params['motion']['slow']['accel_m']
)

r.base.translate_by(
    0.5,
    v_m=r.base.params['motion']['max']['vel_m'],
    a_m=r.base.params['motion']['max']['accel_m']
)

r.end_of_arm.move_to(
    'wrist_yaw',
    1.57,
    v_r=r.end_of_arm.get_joint('wrist_yaw').params['motion']['max']['vel'],
    a_r=r.end_of_arm.get_joint('wrist_yaw').params['motion']['max']['accel']
)
```

# Motion Limits

- hard
- collision
- user
- current

```python
print(r.arm.soft_motion_limits['hard'])   # [0.0, 0.52]  (meters)
print(r.lift.soft_motion_limits['hard'])  # [0.0, 1.1]   (meters)

print(r.end_of_arm.get_joint('wrist_yaw').soft_motion_limits['hard'])    # (radians)
print(r.end_of_arm.get_joint('wrist_pitch').soft_motion_limits['hard'])  # (radians)
print(r.end_of_arm.get_joint('wrist_roll').soft_motion_limits['hard'])   # (radians)
print(r.head.get_joint('head_pan').soft_motion_limits['hard'])           # (radians)
print(r.head.get_joint('head_tilt').soft_motion_limits['hard'])          # (radians)
```

# Data Collection

都以 15 hz 蒐集 (dt = 1/15)

針對 lift, arm 的 5 種 preset 做運動, wrist yaw/pitch/roll, head pan/tilt 的 3 種 preset 做運動

利用 hard motion limit 分成四等分 (得到五個點, 假設是 0, 0.25, 0.5, 0.75, 1)

每次順序是 0 -> 1 -> 0.25 -> 0.75 -> 0.5 (來回) => 四段

每一段直接等 30 seconds 等待動作完成 (總共會是 5  * 4 段 for stepper, 3 * 4 段 for dynamixel)

每次都要存下每個 dt 的 status (pos + vel)

```
calibration_data.h5
├── arm/
│   ├── slow/
│   │   ├── seg_0/   (0.0 → 0.52)
│   │   │   ├── pos  [N,]  float32
│   │   │   └── vel  [N,]  float32
│   │   ├── seg_1/   (0.52 → 0.13)
│   │   ├── seg_2/   (0.13 → 0.39)
│   │   └── seg_3/   (0.39 → 0.26)
│   ├── default/
│   ├── fast/
│   ├── max/
│   └── trajectory_max/
└── lift/
    ├── slow/
    │   ├── seg_0 ...
    ...
```

In [7]:
STEPPER_JOINTS = ["arm", "lift"]
STEPPER_PRESETS = ["slow", "default", "fast", "max", "trajectory_max"]

DXL_JOINTS = ["stretch_gripper", "wrist_yaw", "wrist_pitch", "wrist_roll", "head_pan", "head_tilt"]
DXL_PRESETS     = ["slow", "default", "fast", "max", "trajectory_max"]

In [8]:
# ─────────────────────────────────────────
# Settings
# ─────────────────────────────────────────
DT           = 1 / 15.0
SEG_DURATION = 30.0   # seconds per segment
SETTLE_TIME  = 15.0   # seconds to wait after moving to start pos

current_timestamp = time.time()

def reset():
    """Move all joints to a safe neutral position."""
    robot.arm.move_to(0.2)
    robot.lift.move_to(0.5)
    robot.end_of_arm.move_to("wrist_yaw",   0.0)
    robot.end_of_arm.move_to("wrist_pitch", 0.0)
    robot.end_of_arm.move_to("wrist_roll",  0.0)
    robot.head.move_to("head_pan",  0.0)
    robot.head.move_to("head_tilt", 0.0)
    robot.push_command()
    time.sleep(10.0)

# def get_joint_obj(name):
#     """Return the joint object for a given joint name."""
#     if name in ("arm", "lift"):
#         return getattr(robot, name)
#     elif name in ("wrist_yaw", "wrist_pitch", "wrist_roll", "stretch_gripper"):
#         return robot.end_of_arm.get_joint(name)
#     else:  # head_pan, head_tilt
#         return robot.head.get_joint(name)

def get_joint_obj(name):
    if name in ("arm", "lift"):
        return getattr(robot, name)
    elif name in ("wrist_yaw", "wrist_pitch", "wrist_roll", "stretch_gripper"):
        return robot.end_of_arm.get_joint(name)
    elif name in ("head_pan", "head_tilt"):
        return robot.head.get_joint(name)
    else:
        raise ValueError(f"Unknown joint: {name}")

def get_velocity_keys(joint_name, preset):
    joint = get_joint_obj(joint_name)
    keys = joint.params["motion"][preset]
    for vk, ak in [("vel_m", "accel_m"), ("vel_r", "accel_r"), ("vel", "accel")]:
        if vk in keys:
            return vk, ak
    raise KeyError(f"[{joint_name}/{preset}] unknown keys: {list(keys.keys())}")


def get_waypoints(joint):
    """
    Generate 5 waypoints spanning the full hard range of the joint,
    ordered to exercise the full range:  lo → hi → lo+25% → lo+75% → mid
    """
    lo, hi = joint.soft_motion_limits["hard"]

    if joint.name == "wrist_yaw":
        hi = 3.62
    if joint.name == "wrist_roll":
        hi = 2.839
    if joint.name == "wrist_pitch":
        hi = 0.417
    if joint.name == "head_pan":
        hi = 1.733
    if joint.name == "head_tilt":
        hi = 0.486
    if joint.name == "stretch_gripper":
        hi = 180.0
        lo = -120.0
    
    pts = [lo + i * (hi - lo) / 4 for i in range(5)]
    # 0 → 1 → 0.25 → 0.75 → 0.5
    return [pts[0], pts[4], pts[1], pts[3], pts[2], pts[0]]


def move_to(name, joint, pos, v, a):
    """
    Command joint to absolute position pos with velocity v and acceleration a.

    Stepper: v/a are in m/s  and m/s²  (vel_m / accel_m units)
    DXL:     v/a are in rad/s and rad/s² (world-space, same as 'vel'/'accel' in params)
             move_to accepts them as v_r / a_r
    """
    if name in ("arm", "lift"):
        joint.move_to(pos, v_m=v, a_m=a)
        robot.push_command()
    elif name in ("wrist_yaw", "wrist_pitch", "wrist_roll", "stretch_gripper"):
        robot.end_of_arm.move_to(name, pos, v_r=v, a_r=a)
    elif name in ("head_pan", "head_tilt"): 
        robot.head.move_to(name, pos, v_r=v, a_r=a)
    else:
        raise ValueError(f"Unknown joint: {name}")

def collect_segment(name, joint, target, v, a):
    """
    Command joint to `target`, then record pos & vel for SEG_DURATION seconds.
    Returns (pos_array, vel_array) as float32 numpy arrays.
    """
    move_to(name, joint, target, v, a)
    pos_buf, vel_buf = [], []
    t_end = time.time() + SEG_DURATION
    while time.time() < t_end:
        t0 = time.time()
        robot.pull_status()
        pos_buf.append(joint.status["pos"])
        vel_buf.append(joint.status["vel"])
        elapsed = time.time() - t0
        time.sleep(max(0.0, DT - elapsed))
    return np.array(pos_buf, dtype=np.float32), np.array(vel_buf, dtype=np.float32)


# ─────────────────────────────────────────
# Data collection
# ─────────────────────────────────────────
with h5py.File(f"{current_timestamp}.h5", "w") as f:

    all_joints = (
        # [(name, STEPPER_PRESETS) for name in STEPPER_JOINTS] +
        [(name, DXL_PRESETS)     for name in DXL_JOINTS]
    )

    for joint_name, presets in all_joints:
        joint     = get_joint_obj(joint_name)
        waypoints = get_waypoints(joint)
        segments  = list(itertools.pairwise(waypoints))



        for preset in presets:
            robot.home()
            reset()
            
            vkey, akey = get_velocity_keys(joint_name, preset)
            v = joint.params["motion"][preset][vkey]
            a = joint.params["motion"][preset][akey]
            print(f"\n  [{joint_name}/{preset}]  v={v:.3f}  a={a:.3f}")

            for seg_idx, (start, end) in enumerate(segments):

                # Move to start position, recover from collision if needed
                try:
                    move_to(joint_name, joint, start, v, a)
                except Exception as e:
                    print(f"  [WARN] move_to start failed: {e} → re-homing...")
                    robot.home()
                    reset()
                    move_to(joint_name, joint, start, v, a)

                print(f"  Settling {SETTLE_TIME:.0f}s ...", end=" ", flush=True)
                time.sleep(SETTLE_TIME)

                print(f"seg {seg_idx}: {start:.3f} → {end:.3f}", end=" ... ", flush=True)
                pos, vel = collect_segment(joint_name, joint, end, v, a)

                # Save to HDF5
                grp = f.require_group(f"{joint_name}/{preset}/seg_{seg_idx}")
                grp.create_dataset("pos", data=pos, compression="gzip")
                grp.create_dataset("vel", data=vel, compression="gzip")
                grp.attrs["start"]  = start
                grp.attrs["end"]    = end
                grp.attrs["preset"] = preset
                grp.attrs["v"]      = v
                grp.attrs["a"]      = a

                print(f"saved {len(pos)} samples")

robot.stop()
print("\n✅ Done! Data saved to kf_data.h5")

--------- Homing Head ----
--------- Homing Lift ----
Homing Lift...


[INFO] [robot_monitor]: Guarded contact lift
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event


Hardstop detected at motor position (rad) 115.58843231201172
Marking Lift position to 1.101578 (m)
Marking Lift position to 0.000000 (m)


[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event
[INFO] [robot_monitor]: Base bump event


Lift homing successful
--------- Homing Arm ----
Homing Arm...


...

Hardstop detected at motor position (rad) -0.03892050310969353
Marking Arm position to 0.000000 (m)


[INFO] [robot_monitor]: Wrist single tap: 2322


Arm homing successful
Moving to first hardstop...
First hardstop contact at position (ticks): -5
-----
Homing offset was 5589
Marking current position to zero ticks
Homing offset is now  5588 (ticks)
-----
Current position (ticks): 37
Moving to calibrated zero: (rad)
Moving to first hardstop...
First hardstop contact at position (ticks): -1
-----
Homing offset was 6049
Marking current position to zero ticks
Homing offset is now  6049 (ticks)
-----
Current position (ticks): 26
Moving to calibrated zero: (rad)


[INFO] [robot_monitor]: Wrist single tap: 2324
[INFO] [robot_monitor]: Wrist single tap: 2325



  [head_tilt/trajectory_max]  v=8.000  a=16.000
  Settling 15s ... seg 0: -2.028 → 0.486 ... saved 450 samples
  Settling 15s ... seg 1: 0.486 → -1.399 ... saved 449 samples
  Settling 15s ... seg 2: -1.399 → -0.142 ... saved 449 samples
  Settling 15s ... seg 3: -0.142 → -0.771 ... saved 450 samples
  Settling 15s ... seg 4: -0.771 → -2.028 ... saved 450 samples

✅ Done! Data saved to kf_data.h5


In [17]:
robot.stop()

In [12]:
current_timestamp = "1772787436.1391125"

def get_presets(joint_name):
    return STEPPER_PRESETS if joint_name in STEPPER_JOINTS else DXL_PRESETS


def plot_segment(joint_name, preset, seg_idx):
    with h5py.File(f"{current_timestamp}.h5", "r") as f:
        grp = f[f"{joint_name}/{preset}/seg_{seg_idx}"]
        pos = grp["pos"][:]
        vel = grp["vel"][:]
        start = grp.attrs["start"]
        end = grp.attrs["end"]
        v = grp.attrs.get("v", float("nan"))
        a = grp.attrs.get("a", float("nan"))

    t = np.arange(len(pos)) / 15.0
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=["Position", "Velocity"])
    # fig.add_trace(go.Scatter(x=t, y=pos, mode="lines", name="pos"), row=1, col=1)
    # fig.add_trace(go.Scatter(x=t, y=pos, mode="markers", name="pos"), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=pos, mode="markers", marker=dict(size=3), name="pos"), row=1, col=1)
    # fig.add_trace(go.Scatter(x=t, y=vel, mode="lines", name="vel"), row=2, col=1)
    # fig.add_trace(go.Scatter(x=t, y=vel, mode="markers", name="vel"), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=vel, mode="markers", marker=dict(size=3), name="vel"), row=2, col=1)

    fig.add_hline(y=start, line=dict(color="green", dash="dash"), annotation_text=f"start={start:.3f}", row=1, col=1)
    fig.add_hline(y=end, line=dict(color="red", dash="dash"), annotation_text=f"end={end:.3f}", row=1, col=1)

    fig.add_hline(y=v, line=dict(color="green", dash="dash"), annotation_text=f"v={v:.3f}", row=2, col=1)
    fig.add_hline(y=0, line=dict(color="gray", dash="dot"), annotation_text="0", row=2, col=1)
    
    unit = "m" if joint_name in STEPPER_JOINTS else "rad"
    fig.update_layout(
        title=f"{joint_name} / {preset} / seg_{seg_idx}  ({start:.3f} → {end:.3f} {unit})  v={v:.3f}  a={a:.3f}",
        xaxis2_title="Time (s)",
        yaxis_title=unit,
        yaxis2_title=f"{unit}/s",
        height=500,
        showlegend=False,
    )
    return fig


# widgets
joint_dd = widgets.Dropdown(options=STEPPER_JOINTS + DXL_JOINTS, description="Joint:")
preset_dd = widgets.Dropdown(options=get_presets(joint_dd.value), description="Preset:")
seg_dd = widgets.Dropdown(options=[0, 1, 2, 3], description="Segment:")
output = widgets.Output()


def update_presets(change):
    preset_dd.options = get_presets(change["new"])


def refresh(_=None):
    output.clear_output(wait=True)
    with output:
        plot_segment(joint_dd.value, preset_dd.value, seg_dd.value).show()


joint_dd.observe(update_presets, names="value")
for w in (joint_dd, preset_dd, seg_dd):
    w.observe(refresh, names="value")

display(widgets.HBox([joint_dd, preset_dd, seg_dd]), output)
refresh()

Output()

In [28]:
def analyze_precision(joint_name, preset, seg_idx, t_start, t_end, sample_rate=15.0):
    """
    分析指定時間段內的位置精度統計。
    
    Parameters:
        joint_name : str
        preset     : str
        seg_idx    : int
        t_start    : float  # 秒，相對於 segment 開頭
        t_end      : float  # 秒，相對於 segment 開頭
    
    Returns:
        dict with mean, std, bias, range, and raw samples
    """
    with h5py.File(f"{current_timestamp}.h5", "r") as f:
        grp = f[f"{joint_name}/{preset}/seg_{seg_idx}"]
        pos = grp["pos"][:]
        target = grp.attrs["end"]   # 假設你要比對的目標是 end position

    # 換算成 sample index
    i_start = int(t_start * sample_rate)
    i_end   = int(t_end   * sample_rate)
    samples = pos[i_start:i_end]

    mean   = np.mean(samples)
    std    = np.std(samples)
    bias   = mean - target
    rng    = np.max(samples) - np.min(samples)

    unit = "m" if joint_name in STEPPER_JOINTS else "rad"

    print(f"=== {joint_name} / {preset} / seg_{seg_idx}  [{t_start:.2f}s ~ {t_end:.2f}s] ===")
    print(f"  target : {target:.5f} {unit}")
    print(f"  mean   : {mean:.5f} {unit}")
    print(f"  bias   : {bias:+.5f} {unit}")
    print(f"  std    : {std:.5f} {unit}  (2σ = {2*std:.5f})")
    print(f"  range  : {rng:.5f} {unit}")
    print(f"  suggest ε ≥ {abs(bias) + 3*std:.5f} {unit}  (|bias| + 3σ)")
    print("\n")

    return {
        "mean": mean, "std": std, "bias": bias, "range": rng,
        "target": target, "samples": samples, "unit": unit,
    }

# result = analyze_precision("arm", "slow", seg_idx=0, t_start=12.0, t_end=24.0)
# result = analyze_precision("arm", "slow", seg_idx=1, t_start=12.0, t_end=25.0)

# result = analyze_precision("lift", "fast", seg_idx=0, t_start=10.0, t_end=24.0)

result = analyze_precision("head_tilt", "slow", seg_idx=0, t_start=8.0, t_end=24.0)
result = analyze_precision("head_tilt", "default", seg_idx=0, t_start=8.0, t_end=24.0)
result = analyze_precision("head_tilt", "fast", seg_idx=0, t_start=8.0, t_end=24.0)
result = analyze_precision("head_tilt", "max", seg_idx=0, t_start=8.0, t_end=24.0)
result = analyze_precision("head_tilt", "trajectory_max", seg_idx=0, t_start=8.0, t_end=24.0)

=== head_tilt / slow / seg_0  [8.00s ~ 24.00s] ===
  target : 0.48627 rad
  mean   : 0.48757 rad
  bias   : +0.00130 rad
  std    : 0.00150 rad  (2σ = 0.00300)
  range  : 0.00307 rad
  suggest ε ≥ 0.00580 rad  (|bias| + 3σ)


=== head_tilt / default / seg_0  [8.00s ~ 24.00s] ===
  target : 0.48627 rad
  mean   : 0.48627 rad
  bias   : -0.00000 rad
  std    : 0.00000 rad  (2σ = 0.00000)
  range  : 0.00000 rad
  suggest ε ≥ 0.00000 rad  (|bias| + 3σ)


=== head_tilt / fast / seg_0  [8.00s ~ 24.00s] ===
  target : 0.48627 rad
  mean   : 0.48474 rad
  bias   : -0.00153 rad
  std    : 0.00000 rad  (2σ = 0.00000)
  range  : 0.00000 rad
  suggest ε ≥ 0.00153 rad  (|bias| + 3σ)


=== head_tilt / max / seg_0  [8.00s ~ 24.00s] ===
  target : 0.48627 rad
  mean   : 0.48589 rad
  bias   : -0.00038 rad
  std    : 0.00066 rad  (2σ = 0.00133)
  range  : 0.00153 rad
  suggest ε ≥ 0.00238 rad  (|bias| + 3σ)


=== head_tilt / trajectory_max / seg_0  [8.00s ~ 24.00s] ===
  target : 0.48627 rad
  mean   :